# Exploratory Data Analysis (EDA) & Encoding
Mainly, this notebook covers EDA on the raw trace data and encoding it for use in other steps of the development process.
Also this notebook will compare the performance of using the both method, Which is more robust to go forward without losing any data in the process

In [3]:
import sys
import os
import time
import json
import pandas as pd
import numpy as np
from collections import Counter

sys.path.append("..")

# Import the function from the utils which will needed here
from utils import (
    stream_opcodes,
    count_opcodes_from_stream,
    build_opcode_vocab,
    encode_trace_to_npy,
    save_vocab,
    load_vocab,
    check_vocab_size,
    decode_trace_npy,
    load_encoded_trace
)

## Initializing the Paths

In [4]:
RAW_TRACES_DIR = "../sandbox/data-e2/raw_traces"
MANIFEST_CSV = "../sandbox/data-e2/manifest.csv"

# Load the manifest CSV to get the metadata to use later
df_manifest = pd.read_csv(MANIFEST_CSV)
print(f"Manifest Loaded: {len(df_manifest)} rows")

# Get the list of raw trace files in the directory and sort them for processing
trace_files = sorted([
    os.path.join(RAW_TRACES_DIR, f)
    for f in os.listdir(RAW_TRACES_DIR)
    if f.endswith(".txt")
])
print(f"Raw trace files found: {len(trace_files)}")

Manifest Loaded: 260 rows
Raw trace files found: 260


----
## EDA using Raw Trace Streaming - Method 1
Each of the raw opcode trace file is streamed line by line using `stream_opcodes()` function. A single pass over each file simultaneously to improve the performance.

In [5]:
# Set the data structures to hold the global vocab, per file opcode counts, and streaming times
global_vocab = set()
per_file_counts = {}
stream_times = {}

MIN_OPCODE_COUNT = 10 # Filter threshold to removes likely malformed or truncated opcodes

print(f"Streaming {len(trace_files)} trace files...")
print("This will take a moment base on the given file sizes. The progress shown per 10 files\n")

total_start = time.time()

# Stream through each trace file count the opcodes and store the counts and elapsed time
for i, path in enumerate(trace_files, 1):
    filename = os.path.basename(path)
    t0 = time.time()

    counter = Counter()
    for opcode in stream_opcodes(path):
        counter[opcode] += 1

    elapsed = time.time() - t0
    stream_times[filename] = elapsed
    per_file_counts[filename] = counter

    if i % 10 == 0 or i == len(trace_files):
        done = time.time() - total_start
        print(f"  [{i:>3}/{len(trace_files)}]  elapsed: {done/60:.1f} min")

# Build global counts then filter by minimum threshold
global_counter = Counter()
for c in per_file_counts.values():
    global_counter.update(c)

# Filter the global counter to remove opcodes that are below the minimum threshold
removed = [op for op, count in global_counter.items() if count < MIN_OPCODE_COUNT]
global_vocab = {op for op, count in global_counter.items() if count >= MIN_OPCODE_COUNT}

total_stream_time = time.time() - total_start
print(f"\nStreaming Complete:  {total_stream_time/60:.1f} min total")
print(f"Raw Vocabulary Size: {len(global_counter)} distinct opcodes")
print(f"After Filtering (>={MIN_OPCODE_COUNT}): {len(global_vocab)} distinct opcodes")
print(f"Removed {len(removed)} opcode(s): {removed}")

Streaming 260 trace files...
This will take a moment base on the given file sizes. The progress shown per 10 files

  [ 10/260]  elapsed: 16.0 min
  [ 20/260]  elapsed: 30.3 min
  [ 30/260]  elapsed: 44.1 min
  [ 40/260]  elapsed: 57.8 min
  [ 50/260]  elapsed: 71.9 min
  [ 60/260]  elapsed: 85.9 min
  [ 70/260]  elapsed: 99.1 min
  [ 80/260]  elapsed: 116.7 min
  [ 90/260]  elapsed: 133.8 min
  [100/260]  elapsed: 147.3 min
  [110/260]  elapsed: 160.4 min
  [120/260]  elapsed: 174.5 min
  [130/260]  elapsed: 188.6 min
  [140/260]  elapsed: 204.0 min
  [150/260]  elapsed: 217.7 min
  [160/260]  elapsed: 231.0 min
  [170/260]  elapsed: 246.3 min
  [180/260]  elapsed: 260.5 min
  [190/260]  elapsed: 277.1 min
  [200/260]  elapsed: 293.0 min
  [210/260]  elapsed: 307.6 min
  [220/260]  elapsed: 320.9 min
  [230/260]  elapsed: 334.9 min
  [240/260]  elapsed: 349.4 min
  [250/260]  elapsed: 363.4 min
  [260/260]  elapsed: 379.4 min

Streaming Complete:  379.5 min total
Raw Vocabulary Size: 

### Opcode Vocabulary
The full set of distinct x86 opcode mnemonics seen across all 260 files. `check_vocab_size` confirms the count is safely within the `int32` range used for encoding.

In [6]:
vocab = sorted(global_vocab)
check_vocab_size(vocab)

print(f"Vocabulary Size: {len(vocab)} distinct opcodes")
print(f"Opcodes List: {vocab}")

Vocabulary Size: 147176 distinct opcodes
Opcodes List: ['\x00', '\x00\x00', '\x00MOV', '2MOV', '2S', '2SD', '2SS', 'A', 'A2SD', 'AA', 'AAADD', 'AACMP', 'AAD', 'AADC', 'AADCASTB', 'AADD', 'AADDA', 'AADDADD', 'AADDAND', 'AADDC', 'AADDCALL_NEAR', 'AADDCMP', 'AADDH', 'AADDJ', 'AADDJMP', 'AADDJMPR', 'AADDJNBE', 'AADDJNLE', 'AADDJNZ', 'AADDJZ', 'AADDLEA', 'AADDLEAVE', 'AADDM', 'AADDMOV', 'AADDMOVM', 'AADDMOVSXD', 'AADDMOVZX', 'AADDN', 'AADDNOP', 'AADDNOPR', 'AADDP', 'AADDPOP', 'AADDPUSH', 'AADDR', 'AADDRET_NEAR', 'AADDS', 'AADDSHL', 'AADDSHR', 'AADDSUB', 'AADDT', 'AADDTEST', 'AADDV', 'AADDXOR', 'AADD_LOCK', 'AADD_NEAR', 'AAEA', 'AAEST', 'AAL', 'AALEA', 'AALL_NEAR', 'AAMOV', 'AAMP', 'AAN', 'AAND', 'AANDADD', 'AANDAND', 'AANDC', 'AANDCALL_NEAR', 'AANDCMP', 'AANDH', 'AANDJ', 'AANDJMP', 'AANDJNZ', 'AANDJZ', 'AANDLEA', 'AANDM', 'AANDMOV', 'AANDMOVSXD', 'AANDN', 'AANDNOP', 'AANDNOPR', 'AANDP', 'AANDPOP', 'AANDPUSH', 'AANDRET_NEAR', 'AANDSHR', 'AANDSUB', 'AANDT', 'AANDTEST', 'AANDXOR', 'AANOP', 'AA

### Crosscheck opcode counts with the manifest data

The manifest already has `opcode_count` from the log's "Opcodes recorded" line. Comparing these with the actual line counts from the streaming pass confirms the raw trace files are complete and uncorrupted.

In [7]:
stream_counts = {
    fname: sum(c.values())
    for fname, c in per_file_counts.items()
}

df_cross = df_manifest[["filename", "opcode_count"]].copy()
df_cross["stream_count"] = df_cross["filename"].map(stream_counts)
df_cross["match"] = df_cross["opcode_count"] == df_cross["stream_count"]

mismatches = df_cross[~df_cross["match"].fillna(False)]
print(f"Files with matching counts: {df_cross['match'].sum()}")
print(f"Mismatches or missing count: {len(mismatches)}")

if len(mismatches):
    print("\nMismatched Files:")
    print(mismatches)

Files with matching counts: 0
Mismatches or missing count: 260

Mismatched Files:
                                              filename  opcode_count  \
0    GommeHD.net-Java-API-2.2.0-jar-with-dependenci...     263246339   
1                                     abbot-0.13.0.jar     250721862   
2                              activecluster-4.0.2.jar     192722442   
3                                  activemq-4.0-M1.jar     253984851   
4                               activemq-all-6.2.6.jar     249490122   
..                                                 ...           ...   
255                         xercesjarv-20020712-DG.jar     142176262   
256                               xml-resolver-1.2.jar     190981714   
257                                      xom-1.4.5.jar     184326583   
258                                xsdlib-20050913.jar     199471927   
259                                  ymsg-test-0.6.jar     216474789   

     stream_count  match  
0             NaN  False  

### Global Opcode Frequency Distribution

Analyze the frequency across all 260 files, which opcodes dominate the instruction mix. This clearly shows later feature engineering decisions.

This is a summary table only for this stage, plots will be created on later step.

In [8]:
global_counter = Counter()

for count in per_file_counts.values():
    global_counter.update(count)

df_freq = pd.DataFrame(
    global_counter.most_common(),
    columns=["opcode", "total_count"]
)

df_freq["proportion"] = df_freq["total_count"] / df_freq["total_count"].sum()

print(f"Top 20 most frequent opcodes across all {len(trace_files)} files:\n")
print(df_freq.head(20).to_string(index=False))

Top 20 most frequent opcodes across all 260 files:

   opcode  total_count  proportion
      MOV  16551657666    0.309597
      CMP   4184617496    0.078273
      ADD   2977561280    0.055695
       JZ   2967242707    0.055502
     PUSH   2694236656    0.050395
     TEST   2578068980    0.048222
      POP   2011280005    0.037621
      LEA   1824422652    0.034126
      JNZ   1801158581    0.033690
    MOVZX   1650500473    0.030872
      JMP   1478595408    0.027657
      NOP   1042051557    0.019491
      AND    992052343    0.018556
      SUB    966819900    0.018084
 RET_NEAR    875075676    0.016368
CALL_NEAR    862373616    0.016131
   MOVSXD    643843716    0.012043
    STOSB    581954308    0.010885
      SHR    581647166    0.010880
      XOR    561680447    0.010506


#### Bottom of the distribution to list the rarest opcodes

In [9]:
print("Rarest 20 opcodes (lowest global frequency):\n")
print(df_freq.tail(20).to_string(index=False))

Rarest 20 opcodes (lowest global frequency):

                opcode  total_count   proportion
JZMOVJNLEAXORMOVUPSMPU            1 1.870488e-11
             MPXMOVSXD            1 1.870488e-11
        MOVTESTNOPCMOV            1 1.870488e-11
        ESTCALL_NEARUB            1 1.870488e-11
             NEARJZMOV            1 1.870488e-11
                  PUPO            1 1.870488e-11
            MOVOSHLMOV            1 1.870488e-11
   JNZMOVCALL_NEARRJMP            1 1.870488e-11
            CMPJLEJNZP            1 1.870488e-11
           CMPMOVMOCAL            1 1.870488e-11
      MOVCALL_NEARNJZX            1 1.870488e-11
        ORMOVZXAADDADD            1 1.870488e-11
           TESTJMPJMPJ            1 1.870488e-11
            NOPCPOPUSH            1 1.870488e-11
             ANDSXDEST            1 1.870488e-11
           MOVCJNZSMOV            1 1.870488e-11
              SMOVSHRO            1 1.870488e-11
             VJNLPUSHO            1 1.870488e-11
                  CEQD 

### Check per file trace length distribution

How much do the 260 files vary in total opcode counts, large variation matters for normalization and some file have 260M opcodes and some are 50M opcodes, this should contribute comparable feature vectors, not be dominated by raw size.

In [10]:
df_lengths = pd.DataFrame({
    "filename": list(stream_counts.keys()),
    "opcode_count": list(stream_counts.values()),
})

print(df_lengths["opcode_count"].describe().apply(lambda x: f"{x:,.0f}"))

count            260
mean     205,623,018
std       36,561,706
min      133,264,331
25%      189,135,054
50%      195,242,459
75%      246,254,328
max      276,516,781
Name: opcode_count, dtype: str


### Streaming Timing Summary

Time taken per file using raw text streaming, the baseline for the performance comparison with Method 2 (Encoded Array) later.

In [11]:
df_timing = pd.DataFrame({
    "filename": list(stream_times.keys()),
    "stream_sec": list(stream_times.values()),
})

df_timing["opcodes"] = df_timing["filename"].map(stream_counts)
df_timing["lines_per_sec"] = (df_timing["opcodes"] / df_timing["stream_sec"]).astype(int)

print(f"Streaming Speed Summary (Method 1)")
print(df_timing["lines_per_sec"].describe().apply(lambda x: f"{x:,.0f}"))
print(f"\nTotal Streaming Time: {df_timing['stream_sec'].sum()/60:.1f} min")

Streaming Speed Summary (Method 1)
count          260
mean     2,363,149
std        193,196
min      1,721,871
25%      2,245,891
50%      2,410,806
75%      2,526,487
max      2,601,652
Name: lines_per_sec, dtype: str

Total Streaming Time: 379.4 min


### Save EDA Outputs into CSVs

Save the frequency table and timing results as CSVs so it can be used for later visualization and later notebooks can load them without re-streaming.

In [12]:
os.makedirs("../sandbox/data-e2/processed_outputs", exist_ok=True)

df_freq.to_csv("../sandbox/data-e2/processed_outputs/global_opcode_freq.csv", index=False)
print("global_opcode_freq.csv file saved!")

df_lengths.to_csv("../sandbox/data-e2/processed_outputs/per_file_lengths.csv", index=False)
print("per_file_lengths.csv file saved!")

df_timing.to_csv("../sandbox/data-e2/processed_outputs/streaming_timing.csv", index=False)
print("streaming_timing.csv file saved!")

global_opcode_freq.csv file saved!
per_file_lengths.csv file saved!
streaming_timing.csv file saved!


----
### FIX: Crosscheck opcode counts with the manifest data
The cross-check mismatch is actually a filename extension bug, the manifest.csv has `.jar` filenames but stream_counts keys are `.txt` filenames, so .map() finds nothing. Not a data problem.

In [13]:
# Fix: strip .txt and add .jar to match manifest filename format
stream_counts_jar = {
    fname.replace(".txt", ".jar"): count
    for fname, count in stream_counts.items()
}

df_cross = df_manifest[["filename", "opcode_count"]].copy()
df_cross["stream_count"] = df_cross["filename"].map(stream_counts_jar)
df_cross["match"] = df_cross["opcode_count"] == df_cross["stream_count"]

mismatches = df_cross[~df_cross["match"].fillna(False)]
print(f"Files with matching counts: {df_cross['match'].sum()}")
print(f"Mismatches or missing: {len(mismatches)}")

if len(mismatches):
    print(f"\nMismatched File:\n{mismatches}")

Files with matching counts: 0
Mismatches or missing: 260

Mismatched File:
                                              filename  opcode_count  \
0    GommeHD.net-Java-API-2.2.0-jar-with-dependenci...     263246339   
1                                     abbot-0.13.0.jar     250721862   
2                              activecluster-4.0.2.jar     192722442   
3                                  activemq-4.0-M1.jar     253984851   
4                               activemq-all-6.2.6.jar     249490122   
..                                                 ...           ...   
255                         xercesjarv-20020712-DG.jar     142176262   
256                               xml-resolver-1.2.jar     190981714   
257                                      xom-1.4.5.jar     184326583   
258                                xsdlib-20050913.jar     199471927   
259                                  ymsg-test-0.6.jar     216474789   

     stream_count  match  
0       263122358  False  
1     

The fix actually worked - stream_count is now populated with real numbers, not NaN anymore. There is a tiny difference that is expected and not a problem. 
Because the manifest opcode_count comes from run_trace.sh which uses `wc -l` to count lines. So `wc -l` counts every new line character including empty lines. But the stream_opcodes function skips empty lines. So any empty lines in the .txt file get counted by `wc -l` but skipped by the streamer.

### Fix the crosscheck to use approximate matching instead of exact equal

In [14]:
# Allow a small tolerance value to account for empty lines
TOLERANCE = 0.001

df_cross["diff_pct"] = abs(df_cross["opcode_count"] - df_cross["stream_count"]) / df_cross["opcode_count"]
df_cross["match"] = df_cross["diff_pct"] <= TOLERANCE

mismatches = df_cross[~df_cross["match"].fillna(False)]
print(f"Files within {TOLERANCE*100}% Tolerance: {df_cross['match'].sum()}")
print(f"Files Outside Tolerance: {len(mismatches)}")
print(f"\nMax Difference across all files: {df_cross['diff_pct'].max()*100:.4f}%")
print(f"Mean Difference across all files: {df_cross['diff_pct'].mean()*100:.4f}%")

Files within 0.1% Tolerance: 260
Files Outside Tolerance: 0

Max Difference across all files: 0.0529%
Mean Difference across all files: 0.0440%


### E2 Vocabulary coverage check against the fixed E1 287-opcode vocab

In [15]:
# The vocab is not rebuilt for E2 - it stays fixed
# so feature vectors remain directly comparable across environments
E1_CLEAN_VOCAB = "../sandbox/data/vocab_clean.json"
clean_vocab = load_vocab(E1_CLEAN_VOCAB)
print(f"E1 clean vocab loaded: {len(clean_vocab)} opcodes")

covered = df_freq[df_freq.opcode.isin(clean_vocab)]
total_e2 = df_freq.total_count.sum()
kept_e2 = covered.total_count.sum()

print(f"\nE2 stream covered by fixed 287 vocab: {kept_e2/total_e2*100:.4f}%")
print(f"E2 stream not covered: {(total_e2-kept_e2)/total_e2*100:.4f}%")
print(f"Vocab Entries present in E2: {len(covered)} of {len(clean_vocab)}")

# Opcodes in the fixed vocab that never appear in E2
absent = sorted(set(clean_vocab) - set(covered.opcode))
print(f"\nVocab opcodes absent from E2 ({len(absent)}): {absent}")

# Highest-frequency E2 opcodes NOT in the fixed vocab
missing = df_freq[~df_freq.opcode.isin(clean_vocab)].head(30)
print(f"\nTop 30 E2 opcodes outside the fixed vocab:")
print(missing.to_string(index=False))

E1 clean vocab loaded: 287 opcodes

E2 stream covered by fixed 287 vocab: 99.7860%
E2 stream not covered: 0.2140%
Vocab Entries present in E2: 278 of 287

Vocab opcodes absent from E2 (9): ['LOOP', 'MOVNTPS', 'NOT_LOCK', 'RCR', 'SETNO', 'SHUFPS', 'VMOVHPD', 'VPCMPEQQ', 'VPUNPCKHQDQ']

Top 30 E2 opcodes outside the fixed vocab:
 opcode  total_count  proportion
     OV      4483889    0.000084
 MOVMOV      3989619    0.000075
      V      1889772    0.000035
     MP      1583639    0.000030
     MO      1300810    0.000024
      P      1231626    0.000023
      M      1004911    0.000019
      Z       966208    0.000018
 CMPMOV       954847    0.000018
 MOVCMP       953068    0.000018
     OP       919103    0.000017
    USH       727527    0.000014
    EST       670749    0.000013
PUSHMOV       668218    0.000012
     DD       650121    0.000012
  JZMOV       646635    0.000012
MOVPUSH       624970    0.000012
   MMOV       594225    0.000011
  MOVJZ       587619    0.000011
   MOVM    